# TPC-DS Job Setup

This notebook creates the Databricks workflow jobs for TPC-DS benchmarking.
Connection details are loaded from `config.json` (copy `config.json.example` and fill in your values).

## Three jobs are created

1. **TPC-DS Databricks** — single-combo Databricks pipeline (ingestion + power + concurrency)
2. **TPC-DS Snowflake** — single-combo Snowflake pipeline (ingestion + power + concurrency)
3. **TPC-DS Full Benchmark** — orchestrator. Expands the parameter matrix, generates
   raw data once per scale on a long-lived cluster, runs Databricks and Snowflake
   branches in parallel via `for_each`, and finally ingests results into
   `results_queries` / `results_prices`.

## Widgets (used by this notebook when creating/updating the jobs)

| Widget | Purpose | Default |
|---|---|---|
| `scales` | Comma-separated scale factors in GB | `100` |
| `sizes` | Comma-separated warehouse sizes | `M` |
| `dbx_strategies` | Comma-separated Databricks optimize strategies | `zorder` |
| `sf_strategies` | Comma-separated Snowflake strategies (format+gen) | `native_gen1` |
| `datagen_concurrency` | Parallel data-generation iterations | `2` |
| `dbx_benchmark_concurrency` | Parallel Databricks benchmark iterations | `3` |
| `sf_benchmark_concurrency` | Parallel Snowflake benchmark iterations | `3` |

These values become the **defaults** on the orchestrator job. You can override any of
them at trigger time; `helpers/expand_benchmark_params` validates them and fails
fast with an explicit message if anything is wrong.

## Triggering the orchestrator

All multi-value parameters are **comma-separated**. Do **not** use JSON arrays —
`sizes=["S","M"]` will fail validation.

```bash
# Smoke run (1x1x1x1)
databricks jobs run-now <orchestrator_id>

# Full matrix example: 2 scales x 4 sizes x 3 dbx strategies x 4 sf strategies
databricks jobs run-now <orchestrator_id> \
  --params 'scales=100,1000' \
  --params 'sizes=S,M,L,XL' \
  --params 'dbx_strategies=zorder,liquid_clustering,iceberg_zordered' \
  --params 'sf_strategies=native_gen1,native_gen2,iceberg_gen1,iceberg_gen2'
```

Valid values:
- `sizes` — `2XS`, `XS`, `S`, `M`, `L`, `XL`, `2XL`, `3XL`, `4XL`
- `dbx_strategies` — `zorder`, `liquid_clustering`, `iceberg`, `iceberg_zordered`
- `sf_strategies` — `native_gen1`, `native_gen2`, `iceberg_gen1`, `iceberg_gen2`


In [ ]:
%run "./helpers/Helper Functions"

In [ ]:
# --- Parameters ---
dbutils.widgets.text("scales",         "100",          "Scales (comma-separated GB)")
dbutils.widgets.text("sizes",          "M",            "Warehouse sizes (comma-separated)")
dbutils.widgets.text("dbx_strategies", "zorder",       "Databricks strategies (comma-separated)")
dbutils.widgets.text("sf_strategies",  "native_gen1",  "Snowflake strategies (comma-separated)")
dbutils.widgets.text("datagen_concurrency",        "2", "Data generation for_each concurrency")
dbutils.widgets.text("dbx_benchmark_concurrency",  "3", "Databricks benchmark for_each concurrency")
dbutils.widgets.text("sf_benchmark_concurrency",   "3", "Snowflake benchmark for_each concurrency")
dbutils.widgets.text("max_clusters",               "1", "Max clusters: fixed=MAX_CLUSTER_COUNT, adaptive=QUERY_THROUGHPUT_MULTIPLIER (0 or >=2)")

In [ ]:
import os, json, requests

cfg = load_config()

scales                    = dbutils.widgets.get("scales")
sizes                     = dbutils.widgets.get("sizes")
dbx_strategies            = dbutils.widgets.get("dbx_strategies")
sf_strategies             = dbutils.widgets.get("sf_strategies")
datagen_concurrency       = dbutils.widgets.get("datagen_concurrency")
dbx_benchmark_concurrency = dbutils.widgets.get("dbx_benchmark_concurrency")
sf_benchmark_concurrency  = dbutils.widgets.get("sf_benchmark_concurrency")
max_clusters              = dbutils.widgets.get("max_clusters")

# Defaults baked into the orchestrator job parameters; users can override at trigger time.
base_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
base_path = "/".join(base_path.split("/")[:-1])
host = spark.conf.get("spark.databricks.workspaceUrl")
headers = {"Authorization": f"Bearer {cfg['dbx_pat_token']}"}

print(f"scales             : {scales}")
print(f"sizes              : {sizes}")
print(f"dbx_strategies     : {dbx_strategies}")
print(f"sf_strategies      : {sf_strategies}")
print(f"datagen concurrency: {datagen_concurrency}")
print(f"dbx concurrency    : {dbx_benchmark_concurrency}")
print(f"sf concurrency     : {sf_benchmark_concurrency}")
print(f"base_path          : {base_path}")

JMETER_CLUSTER = {
    "new_cluster": {
        "spark_version": "15.4.x-scala2.12",
        "node_type_id": "m-fleet.4xlarge",
        "num_workers": 0,
        "spark_conf": {
            "spark.master": "local[*]",
            "spark.databricks.cluster.profile": "singleNode",
        },
        "custom_tags": {"ResourceClass": "SingleNode"},
        "data_security_mode": "SINGLE_USER",
        "aws_attributes": {
            "first_on_demand": 1,
            "availability": "SPOT_WITH_FALLBACK",
            "ebs_volume_type": "GENERAL_PURPOSE_SSD",
            "ebs_volume_count": 1,
            "ebs_volume_size": 100,
            "zone_id": "auto",
        },
    }
}

DATAGEN_CLUSTER = {
    "new_cluster": {
        "spark_version": "15.4.x-scala2.12",
        "node_type_id": "rd-fleet.xlarge",
        "autoscale": {"min_workers": 2, "max_workers": 40},
        "data_security_mode": "SINGLE_USER",
        "aws_attributes": {
            "first_on_demand": 1,
            "availability": "SPOT_WITH_FALLBACK",
            "ebs_volume_type": "GENERAL_PURPOSE_SSD",
            "ebs_volume_count": 1,
            "ebs_volume_size": 100,
            "zone_id": "auto",
        },
    }
}

print("JMeter cluster : m-fleet.4xlarge single-node (64GB)")
print("DataGen cluster: rd-fleet.xlarge autoscale 2-40 (persistent across for_each iterations)")

## Verify Raw Data Volume

In [ ]:
# Check that the raw data volume exists for the default scale
vol = cfg["dbx_volume"]
try:
    files = dbutils.fs.ls(f"/Volumes/{cfg['dbx_catalog']}/{cfg['dbx_schema']}/{vol}/{scale}")
    print(f"Volume {vol}: OK ({len(files)} items)")
except Exception as e:
    print(f"WARNING: Volume {vol} not found: {e}")
    print(f"Data generation will create it when the orchestrator runs.")

## Create/Update Jobs

In [ ]:
def make_task(key, notebook, task_params, depends_on=None, cluster="jmeter"):
    task = {
        "task_key": key,
        "notebook_task": {
            "notebook_path": f"{base_path}/{notebook}",
            "base_parameters": task_params,
        },
        "job_cluster_key": cluster,
    }
    if depends_on:
        task["depends_on"] = [{"task_key": d} for d in depends_on]
    return task

def create_or_update_job(name, config):
    resp = requests.get(f"https://{host}/api/2.1/jobs/list?name={name}", headers=headers, timeout=30)
    resp.raise_for_status()
    existing = [j for j in resp.json().get("jobs", []) if j["settings"]["name"] == name]
    if existing:
        job_id = existing[0]["job_id"]
        print(f"  Updating: {name} (ID: {job_id})")
        requests.post(f"https://{host}/api/2.1/jobs/reset", headers=headers,
                      json={"job_id": job_id, "new_settings": config}, timeout=30).raise_for_status()
        return job_id
    else:
        print(f"  Creating: {name}")
        resp = requests.post(f"https://{host}/api/2.1/jobs/create", headers=headers,
                             json=config, timeout=30)
        resp.raise_for_status()
        job_id = resp.json()["job_id"]
        return job_id

# --- Job 1: Databricks Test ---
dbx_test_params = {
    "max_clusters": "{{job.parameters.max_clusters}}",
    "scale": "{{job.parameters.scale}}",
    "warehouse_size": "{{job.parameters.warehouse_size}}",
    "dbx_catalog": cfg["dbx_catalog"],
    "dbx_schema": cfg["dbx_schema"],
    "dbx_volume": cfg["dbx_volume"],
    "dbx_pat_token": cfg["dbx_pat_token"],
    "dbx_optimize_strategy": "{{job.parameters.optimize_strategy}}",
    "dbx_optimize_strategy_slug": "{{job.parameters.optimize_strategy_slug}}",
    "dbx_hourly_rate": "2.00",
    "mlflow_experiment_name": "/TPC-DS Benchmark",
    "benchmark_run_id": "{{job.run_id}}",
}

dbx_job_config = {
    "name": "TPC-DS Databricks",
    "max_concurrent_runs": 5,
    "queue": {"enabled": True},
    "parameters": [
        {"name": "warehouse_size", "default": "M"},
        {"name": "scale", "default": "100"},
        {"name": "optimize_strategy", "default": "zorder"},
        {"name": "optimize_strategy_slug", "default": "ZORDER"},
        {"name": "max_clusters", "default": "1"},
    ],
    "job_clusters": [
        {"job_cluster_key": "jmeter", **JMETER_CLUSTER},
    ],
    "tasks": [
        make_task("query_prep", "2 - Databricks - Ingestion Query Preparation", {
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
            "dbx_optimize_strategy": "{{job.parameters.optimize_strategy}}",
            "scale": "{{job.parameters.scale}}",
            "table_prefix": "{{job.parameters.warehouse_size}}_{{job.parameters.scale}}_{{job.parameters.optimize_strategy_slug}}_",
        }),
        make_task("jmeter_setup", "3 - JMeter Setup", {
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
        }, depends_on=["query_prep"]),
        make_task("jmx_gen", "4 - Databricks - JMX Generation", {
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
            "dbx_warehouse_url": "{{job.parameters.warehouse_size}}",
            "dbx_pat_token": cfg["dbx_pat_token"],
            "dbx_optimize_strategy": "{{job.parameters.optimize_strategy}}",
            "dbx_optimize_strategy_slug": "{{job.parameters.optimize_strategy_slug}}",
            "scale": "{{job.parameters.scale}}",
            "table_prefix": "{{job.parameters.warehouse_size}}_{{job.parameters.scale}}_{{job.parameters.optimize_strategy_slug}}_",
        }, depends_on=["jmeter_setup"]),
        make_task("ingestion", "5a - Databricks - Ingestion", dbx_test_params, depends_on=["jmx_gen"]),
        make_task("power_test", "5b - Databricks - Power Test", dbx_test_params, depends_on=["ingestion"]),
        make_task("concurrency", "5c - Databricks - Concurrency", dbx_test_params, depends_on=["power_test"]),
    ],
}

dbx_job_id = create_or_update_job(dbx_job_config["name"], dbx_job_config)

# --- Job 2: Snowflake Test ---
sf_test_params = {
    "max_clusters": "{{job.parameters.max_clusters}}",
    "scale": "{{job.parameters.scale}}",
    "warehouse_size": "{{job.parameters.warehouse_size}}",
    "dbx_catalog": cfg["dbx_catalog"],
    "dbx_schema": cfg["dbx_schema"],
    "dbx_volume": cfg["dbx_volume"],
    "sf_account_locator": cfg["sf_account_locator"],
    "sf_region": cfg["sf_region"],
    "sf_database": cfg["sf_database"],
    "sf_schema": cfg["sf_schema"],
    "sf_role": cfg["sf_role"],
    "sf_user": cfg["sf_user"],
    "sf_hourly_rate": "4.00",
    "sf_strategy": "{{job.parameters.sf_strategy}}",
    "sf_strategy_slug": "{{job.parameters.sf_strategy_slug}}",
    "mlflow_experiment_name": "/TPC-DS Benchmark",
    "benchmark_run_id": "{{job.run_id}}",
}

sf_job_config = {
    "name": "TPC-DS Snowflake",
    "max_concurrent_runs": 5,
    "queue": {"enabled": True},
    "parameters": [
        {"name": "warehouse_size", "default": "M"},
        {"name": "scale", "default": "100"},
        {"name": "sf_strategy", "default": "native_gen1"},
        {"name": "sf_strategy_slug", "default": "NATIVE_GEN1"},
        {"name": "max_clusters", "default": "1"},
    ],
    "job_clusters": [
        {"job_cluster_key": "jmeter", **JMETER_CLUSTER},
    ],
    "tasks": [
        make_task("query_prep", "2.1 - Snowflake - Data Ingestion", {
            "sf_database": cfg["sf_database"],
            "sf_schema": cfg["sf_schema"],
            "sf_staging": cfg["sf_staging"],
            "scale": "{{job.parameters.scale}}",
            "table_prefix": "{{job.parameters.warehouse_size}}_{{job.parameters.scale}}_{{job.parameters.sf_strategy_slug}}_",
            "sf_strategy": "{{job.parameters.sf_strategy}}",
            "sf_strategy_slug": "{{job.parameters.sf_strategy_slug}}",
            "sf_external_volume": cfg["sf_external_volume"],
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
        }),
        make_task("jmeter_setup", "3 - JMeter Setup", {
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
        }, depends_on=["query_prep"]),
        make_task("jmx_gen", "4.1- Snowflake - JMX Generation", {
            "scale": "{{job.parameters.scale}}",
            "sf_account_locator": cfg["sf_account_locator"],
            "sf_region": cfg["sf_region"],
            "sf_warehouse": "{{job.parameters.warehouse_size}}",
            "sf_database": cfg["sf_database"],
            "sf_schema": cfg["sf_schema"],
            "sf_role": cfg["sf_role"],
            "sf_user": cfg["sf_user"],
            "sf_auth_key": cfg["sf_auth_key"],
            "sf_strategy": "{{job.parameters.sf_strategy}}",
            "sf_strategy_slug": "{{job.parameters.sf_strategy_slug}}",
            "table_prefix": "{{job.parameters.warehouse_size}}_{{job.parameters.scale}}_{{job.parameters.sf_strategy_slug}}_",
            "dbx_catalog": cfg["dbx_catalog"],
            "dbx_schema": cfg["dbx_schema"],
            "dbx_volume": cfg["dbx_volume"],
        }, depends_on=["jmeter_setup"]),
        make_task("ingestion", "5.1a - Snowflake - Ingestion", sf_test_params, depends_on=["jmx_gen"]),
        make_task("power_test", "5.1b - Snowflake - Power Test", sf_test_params, depends_on=["ingestion"]),
        make_task("concurrency", "5.1c - Snowflake - Concurrency", sf_test_params, depends_on=["power_test"]),
    ],
}

sf_job_id = create_or_update_job(sf_job_config["name"], sf_job_config)

print(f"\n  Databricks job: {dbx_job_id}")
print(f"  Snowflake job:  {sf_job_id}")

In [ ]:
# --- Job 3: Orchestrator ---
# Flow:
#   param_expand -> data_generation (for_each over scales)
#                -> databricks_tests (for_each over dbx_combos)  in parallel with
#                -> snowflake_tests  (for_each over sf_combos)
orchestrator_config = {
    "name": "TPC-DS Full Benchmark",
    "parameters": [
        {"name": "scales",                    "default": scales},
        {"name": "sizes",                     "default": sizes},
        {"name": "dbx_strategies",            "default": dbx_strategies},
        {"name": "sf_strategies",             "default": sf_strategies},
        {"name": "datagen_concurrency",       "default": datagen_concurrency},
        {"name": "dbx_benchmark_concurrency", "default": dbx_benchmark_concurrency},
        {"name": "sf_benchmark_concurrency",  "default": sf_benchmark_concurrency},
        {"name": "max_clusters",              "default": max_clusters},
    ],
    "job_clusters": [
        {"job_cluster_key": "datagen", **DATAGEN_CLUSTER},
        {"job_cluster_key": "jmeter",  **JMETER_CLUSTER},
    ],
    # Serverless env for tiny no-op tasks (avoids cluster warmup latency)
    "environments": [
        {"environment_key": "serverless", "spec": {"client": "2"}},
    ],
    "tasks": [
        # Step 0: Expand parameters into Cartesian-product task values
        {
            "task_key": "param_expand",
            "notebook_task": {
                "notebook_path": f"{base_path}/helpers/expand_benchmark_params",
                "base_parameters": {
                    "scales":                    "{{job.parameters.scales}}",
                    "sizes":                     "{{job.parameters.sizes}}",
                    "dbx_strategies":            "{{job.parameters.dbx_strategies}}",
                    "sf_strategies":             "{{job.parameters.sf_strategies}}",
                    "datagen_concurrency":       "{{job.parameters.datagen_concurrency}}",
                    "dbx_benchmark_concurrency": "{{job.parameters.dbx_benchmark_concurrency}}",
                    "sf_benchmark_concurrency":  "{{job.parameters.sf_benchmark_concurrency}}",
                    "max_clusters":              "{{job.parameters.max_clusters}}",
                    "dbx_child_job_name":        "TPC-DS Databricks",
                    "sf_child_job_name":         "TPC-DS Snowflake",
                    "dbx_catalog":               cfg["dbx_catalog"],
                    "dbx_schema":                cfg["dbx_schema"],
                    "dbx_volume":                cfg["dbx_volume"],
                },
            },
            "environment_key": "serverless",
        },

        # Step 1: Generate raw data per scale on a single long-lived cluster
        {
            "task_key": "data_generation",
            "for_each_task": {
                "inputs": "{{tasks.param_expand.values.scales_to_generate_json}}",
                "concurrency": int(datagen_concurrency),
                "task": {
                    "task_key": "datagen_iter",
                    "notebook_task": {
                        "notebook_path": f"{base_path}/1 - Raw Data Generator",
                        "base_parameters": {
                            "dbx_catalog": cfg["dbx_catalog"],
                            "dbx_schema": cfg["dbx_schema"],
                            "dbx_volume": cfg["dbx_volume"],
                            "dbx_volume_location": "",
                            "scale": "{{input}}",
                        },
                    },
                    "job_cluster_key": "datagen",
                },
            },
            "depends_on": [{"task_key": "param_expand"}],
        },

        # Step 2: Databricks benchmark per combo (parallel across combos)
        {
            "task_key": "databricks_tests",
            "for_each_task": {
                "inputs": "{{tasks.param_expand.values.dbx_combos_json}}",
                "concurrency": int(dbx_benchmark_concurrency),
                "task": {
                    "task_key": "dbx_test",
                    "run_job_task": {
                        "job_id": dbx_job_id,
                        "job_parameters": {
                            "scale":                  "{{input.scale}}",
                            "warehouse_size":         "{{input.size}}",
                            "optimize_strategy":      "{{input.strategy_raw}}",
                            "optimize_strategy_slug": "{{input.strategy_slug}}",
                            "max_clusters":           "{{job.parameters.max_clusters}}",
                        },
                    },
                },
            },
            "depends_on": [{"task_key": "data_generation"}],
        },

        # Step 3: Snowflake benchmark per combo (parallel across combos; parallel with Databricks branch)
        {
            "task_key": "snowflake_tests",
            "for_each_task": {
                "inputs": "{{tasks.param_expand.values.sf_combos_json}}",
                "concurrency": int(sf_benchmark_concurrency),
                "task": {
                    "task_key": "sf_test",
                    "run_job_task": {
                        "job_id": sf_job_id,
                        "job_parameters": {
                            "scale":            "{{input.scale}}",
                            "warehouse_size":   "{{input.size}}",
                            "sf_strategy":      "{{input.strategy_raw}}",
                            "sf_strategy_slug": "{{input.strategy_slug}}",
                            "max_clusters":     "{{job.parameters.max_clusters}}",
                        },
                    },
                },
            },
            "depends_on": [{"task_key": "data_generation"}],
        },

        # Step 4: Ingest benchmark result CSVs + update derived price table.
        # Autoloader checkpoint on the volume -> only new files are processed.
        {
            "task_key": "results_ingestion",
            "notebook_task": {
                "notebook_path": f"{base_path}/6 - Results Comparison",
                "base_parameters": {
                    "dbx_catalog": cfg["dbx_catalog"],
                    "dbx_schema":  cfg["dbx_schema"],
                    "dbx_volume":  cfg["dbx_volume"],
                    "region":      cfg.get("aws_region", "us-east-1"),
                },
            },
            "environment_key": "serverless",
            "depends_on": [
                {"task_key": "databricks_tests"},
                {"task_key": "snowflake_tests"},
            ],
        },

        # Step 5: Refresh AWS pricing for both platforms so the dashboard's
        # cost columns reflect current list prices. Idempotent — MERGEs over
        # the existing rows in `results_prices`.
        {
            "task_key": "pricing_update",
            "notebook_task": {
                "notebook_path": f"{base_path}/6.1 - Pricing Update",
            },
            "environment_key": "serverless",
            "depends_on": [{"task_key": "results_ingestion"}],
        },
    ],
}

orchestrator_job_id = create_or_update_job(orchestrator_config["name"], orchestrator_config)

print(f"\n{'='*60}")
print(f"  JOBS CREATED/UPDATED")
print(f"{'='*60}")
print(f"  1. TPC-DS Databricks      ID: {dbx_job_id}")
print(f"  2. TPC-DS Snowflake       ID: {sf_job_id}")
print(f"  3. TPC-DS Full Benchmark  ID: {orchestrator_job_id}")
print(f"{'='*60}")
print()
print("Smoke run (1 scale x 1 size x 1 strategy each):")
print(f"  databricks jobs run-now {orchestrator_job_id}")
print()
print("Wider run example:")
print(f"  databricks jobs run-now {orchestrator_job_id} \\\n"
      f"    --params 'scales=100,1000' \\\n"
      f"    --params 'sizes=S,M,L,XL' \\\n"
      f"    --params 'dbx_strategies=zorder,liquid_clustering,iceberg_zordered' \\\n"
      f"    --params 'sf_strategies=native_gen1,native_gen2,iceberg_gen1,iceberg_gen2'")